# 8 比特量子核：纯解析输入的最小示例

本 Notebook 展示线路、精确状态、保真度核、中心化回归对齐、KRR 和有限测量下的最终选择。输入由固定随机种子构造，不含金融历史或实验室数据。它是组成环节的教学示例；完整的布局标签、MLP 筛选和微调流程由 `run_search.py` 承担。

使用本模块锁定的 PennyLane 0.33.1 环境。本文档不自动安装依赖、不联网、不提交云任务。所有单元在发布时均保留为空输出；运行后的数值才是本次运行记录。


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pennylane as qml

here = Path.cwd().resolve()
source = next((p / 'experiments' / 'qukernet_finance' for p in [here, *here.parents]
               if (p / 'experiments' / 'qukernet_finance' / 'quantum_engine.py').is_file()), None)
if source is None:
    raise RuntimeError('请从本课程仓库内启动 Notebook。')
sys.path.insert(0, str(source))
import quantum_engine as qe
from modeling import krr
from data_pipeline import private_output

assert qml.__version__ == '0.33.1', '请使用已冻结环境；本 Notebook 不会自行安装依赖。'
print({'PennyLane': qml.__version__, 'NumPy': np.__version__})


## 1. 构造输入与两个固定候选

角度直接传入引擎，不再乘以 π。前 8 个样本用于训练，后 4 个只用于本示例的验证。连续目标依赖所有 8 个输入；目标的均值和标准差只由训练部分计算。这是解析函数回归，不是股票收益。

每个 block 先执行奇偶线上的 RX/RY，再依次执行相邻 CNOT。最后共同的纠缠层在保真度中抵消；深度 2 在此前纠缠后重新编码，因此可以改变核。实际是否不同仍由核值检查。


In [ ]:
rng = np.random.default_rng(20261109)
x_angles = rng.uniform(-0.9, 0.9, size=(12, 8))
y_raw = np.sin(x_angles).mean(axis=1) + 0.1 * (x_angles[:, :-1] * x_angles[:, 1:]).mean(axis=1)
y_mean, y_std = y_raw[:8].mean(), y_raw[:8].std(ddof=1)
assert y_std > 1e-12
y = (y_raw - y_mean) / y_std
n_train, n_valid = 8, 4
layouts = [
    {'q': 8, 'depth': depth, 'axes': [0, 1], 'edges': [1]*7, 'id': f'demo_depth_{depth}'}
    for depth in (1, 2)
]
print({'canonical_search_layouts': len(qe.all_layouts()), 'demo_candidates': layouts})

# 仅为显示复用引擎内部的同一 QNode，不另写线路模拟器。
node = qe._state_qnode(qe._layout_key(layouts[1]), ())
print(qml.draw(node)(x_angles[0], np.array([], dtype=float)))


## 2. 状态、核与中心化对齐

精确状态由 `default.qubit` 产生，采用 complex128。核元素为两个状态内积的模平方。回归对齐是中心化训练核与中心化目标外积的 Frobenius 对齐；它是候选质量代理，不是验证 MSE、排名效果或交易收益。


In [ ]:
cached_states = [qe.states(x_angles, layout) for layout in layouts]
exact_kernels = [qe.gram(state) for state in cached_states]
exact_train = [K[:n_train, :n_train] for K in exact_kernels]
exact_cross = [K[n_train:, :n_train] for K in exact_kernels]
for layout, state, K in zip(layouts, cached_states, exact_train):
    print({'id': layout['id'], 'state_dtype': str(state.dtype),
           'max_norm_error': float(np.max(np.abs(np.sum(np.abs(state)**2, axis=1)-1))),
           'diagonal_error': float(np.max(np.abs(np.diag(K)-1))),
           'training_alignment': qe.alignment(K, y[:n_train])})
print({'candidate_kernel_max_difference': float(np.max(np.abs(exact_train[0]-exact_train[1])))})


## 3. 下游 KRR 与精确评价器

复用课程规定的训练谱处理、相容交叉核延拓、训练中心化与尺度。正则化固定为 λ=0.1，系统是 `K + nλI`。此处的精确验证损失保留给后面的事后评价器，采样选择函数不会接收它。


In [ ]:
regularization = 0.1
exact_losses = []
for layout, K, B in zip(layouts, exact_train, exact_cross):
    prediction, diagnostics = krr(K, B, y[:n_train], regularization)
    loss = float(np.mean((prediction-y[n_train:])**2))
    exact_losses.append(loss)
    print({'id': layout['id'], 'exact_validation_mse': loss,
           'retained_rank': diagnostics['retained_rank']})


## 4. 有限 shots：采样器与选择器分开

采样器以精确概率产生理想二项计数。选择器只接收计数、shots、标签和固定 λ；每次重做谱处理及 KRR 拟合。训练核对角线在理想保真度模型下固定为 1，上三角和镜像共用同一计数。不同候选的读数独立。

这只模拟理想有限测量误差，不代表器件噪声或任何云执行。下面没有根据精确验证损失调参或替换候选。


In [ ]:
def sample_counts(K, B, shots, seed):
    if min(K.min(), B.min()) < -1e-12 or max(K.max(), B.max()) > 1+1e-12:
        raise ValueError('精确概率越界，须先检查核实现。')
    local_rng = np.random.default_rng(seed)
    rows, cols = np.triu_indices(len(K), 1)
    train_counts = np.zeros(K.shape, dtype=np.int64)
    successes = local_rng.binomial(shots, np.clip(K[rows, cols], 0, 1))
    train_counts[rows, cols] = successes
    train_counts[cols, rows] = successes
    np.fill_diagonal(train_counts, shots)
    cross_counts = local_rng.binomial(shots, np.clip(B, 0, 1))
    return {'train': train_counts, 'cross': cross_counts}


def choose_from_counts(counts, shots, train_y, valid_y, lam):
    losses, failures = [], []
    for candidate_counts in counts:
        try:
            prediction, _ = krr(candidate_counts['train']/shots,
                                candidate_counts['cross']/shots, train_y, lam)
            loss = float(np.mean((prediction-valid_y)**2))
            if not np.isfinite(loss):
                raise ValueError('nonfinite validation loss')
            losses.append(loss)
            failures.append(None)
        except (ValueError, np.linalg.LinAlgError) as error:
            losses.append(None)
            failures.append(type(error).__name__ + ': ' + str(error))
    eligible = [i for i, loss in enumerate(losses) if loss is not None]
    # 数值完全并列时按固定候选顺序取先者。
    chosen = min(eligible, key=lambda i: (losses[i], i)) if eligible else None
    return {'selected': chosen, 'estimated_losses': losses, 'failures': failures}


## 5. 小规模重复与资源账本

每档仅 5 次重复，用于看懂代码和观察现象，不足以给出可靠性研究结论。简单遗憾比较所选候选与精确最佳候选的损失；它不包含所选核内部的额外拟合误差。预算中的 shots 是理想采样名义数，不是实际云额度或真实量子测量次数。


In [ ]:
repetitions = 5
records = []
for shots in (32, 128, 512):
    for repeat in range(repetitions):
        counts = [sample_counts(K, B, shots,
                                np.random.SeedSequence([20261109, shots, repeat, candidate]))
                  for candidate, (K, B) in enumerate(zip(exact_train, exact_cross))]
        decision = choose_from_counts(counts, shots, y[:n_train], y[n_train:], regularization)
        chosen = decision['selected']
        # 从这一行开始才允许事后评价器读取精确参照。
        regret = None if chosen is None else float(exact_losses[chosen]-min(exact_losses))
        records.append({'shots_per_action': shots, 'repeat': repeat,
                        'selected': chosen, 'simple_regret': regret,
                        'estimated_losses': decision['estimated_losses'],
                        'failures': decision['failures']})

actions = len(layouts) * (n_train*(n_train-1)//2 + n_train*n_valid)
ledger = {'actions_per_repetition': actions,
          'nominal_shots': actions*repetitions*sum((32, 128, 512)),
          'real_cloud_shots': 0, 'cloud_jobs': 0,
          'cached_exact_state_rows': len(layouts)*len(x_angles)}
print(json.dumps({'ledger': ledger, 'records': records}, indent=2))


## 6. 用户选择私有输出目录后才保存

默认不保存任何实验文件。若要留存运行记录，将 `PRIVATE_OUTPUT` 改成自己的本地私有目录；目录不能位于公开课程源码内，也不能是网络共享位置。已有同名结果会报错，不覆盖旧运行。这里不会保存完整状态向量。


In [ ]:
PRIVATE_OUTPUT = None  # 用户自行填写本地私有目录字符串。
if PRIVATE_OUTPUT is None:
    print('未选择输出目录；本次不写文件。')
else:
    destination = private_output(PRIVATE_OUTPUT, script_path=source/'quantum_engine.py')
    payload = {'kind': 'analytic_8q_teaching_demo', 'seed': 20261109,
               'pennylane_version': qml.__version__, 'layouts': layouts,
               'regularization': regularization, 'exact_losses_for_posthoc_only': exact_losses,
               'ledger': ledger, 'records': records}
    with (destination/'analytic_8q_demo.json').open('x', encoding='utf-8') as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2, allow_nan=False)
    print('已保存教学运行记录。')


## 阅读与扩展

- `verify_engine.py` 提供独立重叠线路、两比特矩阵参考和梯度检查；其通过状态以实际运行记录为准。
- `run_search.py` 承担完整的回归迁移搜索；`reproduction_differences.csv` 逐项列出与论文及公开代码的区别。
- [QuKerNet ICLR 2024 正式版](https://proceedings.iclr.cc/paper_files/paper/2024/file/4d9944ab3330fe6af8efb9260aa9f307-Paper-Conference.pdf)，§3.1 与 Appendix B。
- [公开代码固定版本](https://github.com/iamconglei-ctrl/QuKerNet/tree/c9948606145eb85c2bf53f27acebaf5a071a3fa7)。公开代码示例参数与论文实验配置分别记载，不混作完全相同的实验。
